# Main Analysis: International Student Enrolments and Melbourne Vacancy Rates

This notebook provides the main econometric analysis for the project. It uses the cleaned quarterly dataset from 2020 Q1 to 2025 Q4 to examine whether Victoria international student enrolments are descriptively associated with estimated Melbourne rental vacancy rates.

The analysis is descriptive rather than causal. It examines conditional correlations and predictive associations, not an identified treatment effect.

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

df = pd.read_csv("../data/clean/clean_data.csv")
df["quarter_end_date"] = pd.to_datetime(df["quarter_end_date"])

df.head()

,year,quarter,quarter_end_month,quarter_end_date,vic_ytd_enrolments,vic_ytd_commencements,estimated_melbourne_vacancies,estimated_melbourne_vacancy_rate_pct,student_geography,vacancy_geography,notes,quarter_num,quarter_end_month_num
0,2020,Q1,Mar,2020-03-31,216681,56991,3000,4.4,VIC,Melbourne,Quarter-end YTD student data; vacancy is chart...,1,3
1,2020,Q2,Jun,2020-06-30,235433,75743,6200,8.0,VIC,Melbourne,Quarter-end YTD student data; vacancy is chart...,2,6
2,2020,Q3,Sep,2020-09-30,267172,107482,8100,10.3,VIC,Melbourne,Quarter-end YTD student data; vacancy is chart...,3,9
3,2020,Q4,Dec,2020-12-31,282720,123030,7800,9.9,VIC,Melbourne,Quarter-end YTD student data; vacancy is chart...,4,12
4,2021,Q1,Mar,2021-03-31,178296,37476,6700,8.5,VIC,Melbourne,Quarter-end YTD student data; vacancy is chart...,1,3


## Analysis declaration

This analysis is descriptive, not causal. It examines whether Victoria international student enrolments are associated with estimated Melbourne rental vacancy rates from 2020 Q1 to 2025 Q4.

The analysis does not attempt to identify a causal treatment effect. The main reason is that the data are observational, the student variable is measured at the Victoria level while the vacancy-rate variable refers to Melbourne, and the vacancy-rate values are approximated from SQM public charts.

## Econometric specification

The main specification is:

\[
VacancyRate_t = \beta_0 + \beta_1 Enrolments100k_t + \varepsilon_t
\]

where:

- \(VacancyRate_t\) is the estimated Melbourne rental vacancy rate in quarter \(t\).
- \(Enrolments100k_t\) is Victoria year-to-date international student enrolments divided by 100,000.
- \(\varepsilon_t\) is the regression error term.

The coefficient \(\beta_1\) measures the descriptive association between an additional 100,000 Victoria international student enrolments and the estimated Melbourne vacancy rate, measured in percentage points.

Because this is quarterly time-series data, the regression uses OLS with HAC standard errors allowing for serial correlation up to one quarter. The sample covers 2020 Q1 to 2025 Q4.

In [3]:
# Prepare variables for regression
df_analysis = df.copy()

# Scale enrolments so the coefficient is easier to interpret
df_analysis["enrolments_100k"] = df_analysis["vic_ytd_enrolments"] / 100000

# Keep only variables needed for the main model
df_analysis = df_analysis[
    [
        "year",
        "quarter",
        "quarter_end_date",
        "estimated_melbourne_vacancy_rate_pct",
        "vic_ytd_enrolments",
        "enrolments_100k",
    ]
].dropna()

df_analysis.head()

,year,quarter,quarter_end_date,estimated_melbourne_vacancy_rate_pct,vic_ytd_enrolments,enrolments_100k
0,2020,Q1,2020-03-31,4.4,216681,2.16681
1,2020,Q2,2020-06-30,8.0,235433,2.35433
2,2020,Q3,2020-09-30,10.3,267172,2.67172
3,2020,Q4,2020-12-31,9.9,282720,2.82720
4,2021,Q1,2021-03-31,8.5,178296,1.78296


In [4]:
# Main OLS regression with HAC standard errors
y = df_analysis["estimated_melbourne_vacancy_rate_pct"]
X = df_analysis[["enrolments_100k"]]
X = sm.add_constant(X)

main_model = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": 1})

print(main_model.summary())

                                     OLS Regression Results                                     
Dep. Variable:     estimated_melbourne_vacancy_rate_pct   R-squared:                       0.000
Model:                                              OLS   Adj. R-squared:                 -0.045
Method:                                   Least Squares   F-statistic:                 0.0006785
Date:                                  Tue, 12 May 2026   Prob (F-statistic):              0.979
Time:                                          19:03:23   Log-Likelihood:                -59.107
No. Observations:                                    24   AIC:                             122.2
Df Residuals:                                        22   BIC:                             124.6
Df Model:                                             1                                         
Covariance Type:                                    HAC                                         
                      coef    

In [5]:
# Create a clean regression table

regression_table = pd.DataFrame({
    "Specification": ["Main OLS"],
    "Dependent variable": ["Estimated Melbourne vacancy rate (%)"],
    "Regressor": ["Victoria YTD enrolments / 100,000"],
    "Coefficient": [main_model.params["enrolments_100k"]],
    "Standard error": [main_model.bse["enrolments_100k"]],
    "N": [int(main_model.nobs)],
    "R-squared": [main_model.rsquared],
    "Standard errors": ["HAC, 1 lag"]
})

regression_table["Coefficient"] = regression_table["Coefficient"].round(3)
regression_table["Standard error"] = regression_table["Standard error"].round(3)
regression_table["R-squared"] = regression_table["R-squared"].round(3)

regression_table

,Specification,Dependent variable,Regressor,Coefficient,Standard error,N,R-squared,Standard errors
0,Main OLS,Estimated Melbourne vacancy rate (%),"Victoria YTD enrolments / 100,000",0.03,1.162,24,0.0,"HAC, 1 lag"


In [6]:
import os

os.makedirs("../outputs/tables", exist_ok=True)

regression_table.to_csv("../outputs/tables/regression_table_quarterly.csv", index=False)

print("Saved regression table to ../outputs/tables/regression_table_quarterly.csv")

Saved regression table to ../outputs/tables/regression_table_quarterly.csv


## Interpretation of the main coefficient

The estimated coefficient on `enrolments_100k` is 0.030. This means that an additional 100,000 Victoria year-to-date international student enrolments is associated with a 0.030 percentage point increase in the estimated Melbourne vacancy rate.

This estimated association is extremely small in magnitude. The HAC standard error is 1.162, which is much larger than the coefficient estimate, and the p-value from the regression output is 0.979. The R-squared is approximately 0.000, meaning that contemporaneous international student enrolments explain almost none of the variation in the estimated Melbourne vacancy rate in this sample.

Therefore, the main regression does not provide evidence of a meaningful contemporaneous association between Victoria international student enrolments and estimated Melbourne rental vacancy rates.

## Threats and limitations

This analysis is descriptive, so the main concern is not failure of a causal identification strategy, but whether the observed association is informative about the relationship between international student demand and rental-market conditions.

Several limitations are important.

First, there is a geographic mismatch. The international student variable is measured at the Victoria level, while the vacancy-rate variable refers to Melbourne. If international student changes outside Melbourne are not closely related to Melbourne rental demand, this mismatch may weaken the estimated relationship.

Second, the vacancy-rate data are approximate. The Melbourne vacancy-rate series was visually estimated from public SQM Research charts because the underlying downloadable data require purchase. This introduces measurement error in the outcome variable.

Third, the sample period from 2020 to 2025 is strongly affected by COVID-era disruptions, including border closures, lockdowns, migration changes, and unusual rental-market conditions. These shocks may dominate the simple relationship between student enrolments and vacancy rates.

Fourth, the model does not include controls for other determinants of vacancy rates, such as housing supply, domestic migration, interest rates, rents, employment conditions, or broader population growth. These omitted variables may confound the descriptive association.

Fifth, the enrolments variable is measured as year-to-date enrolments at quarter-end months. This is not the same as the number of unique students living in Melbourne in each quarter, and it may not capture the timing of actual rental demand precisely.

For these reasons, the regression should be interpreted as a descriptive correlation only. The results do not identify the causal effect of international students on Melbourne vacancy rates.

## Conclusion

The main analysis finds no meaningful contemporaneous association between Victoria international student enrolments and estimated Melbourne rental vacancy rates over 2020 Q1 to 2025 Q4. The estimated coefficient is close to zero, statistically insignificant, and the model has an R-squared close to zero.

This does not rule out a relationship between international students and rental-market conditions. Instead, it suggests that the simple contemporaneous quarterly association is weak in the current data. More precise analysis would require geographically matched student and housing data, more accurate vacancy-rate data, and controls for other major rental-market factors.